<a href="https://colab.research.google.com/github/jzhino18/ds4420_finalrepo/blob/main/poc-train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## DS 4420 - Proof of Concept (Model Training)

- Key Question: To what extent can a game's content, quality, and engagement metrics be used to predict its price on the Steam marketplace?
- File purpose: Model training for POC
- Target variable: `Price` (regression)
- Dataset used in this notebook: `steam_clean.csv` (40,677 rows)

### Current RMSE Snapshot (same 80/20 split)
- Baseline notebook MLP: `3.388250`
- Linear/Ridge: `0.405974`
- Best model (`HistGradientBoostingRegressor`): `0.341865`

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [7]:
# pulling the df we cleaned earlier
steam = pd.read_csv('steam_clean.csv')
steam

,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,Windows,Mac,Linux,Metacritic score,User score,Positive,Negative,Achievements,Recommendations,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,language_count
0,0.000000,0.0,5.24,4.189655,0,1,0,0,0,0,5.533389,1.386294,0,5.446737,2.197225,0.000000,2.197225,0.000000,0
1,2.197225,0.0,35.99,2.397895,1,1,0,0,0,0,4.770685,2.639057,0,4.682131,6.516193,0.000000,7.134094,0.000000,0
2,0.000000,0.0,1.49,4.262680,0,1,1,0,0,0,4.174387,1.098612,13,0.000000,0.000000,0.000000,0.000000,0.000000,0
3,0.000000,0.0,1.35,4.204693,0,1,0,0,0,0,3.555348,2.397895,5,0.000000,0.000000,0.000000,0.000000,0.000000,0
4,4.174387,0.0,2.99,4.394449,1,1,1,0,0,0,9.338558,6.723832,11,9.328212,5.837730,4.465908,5.159055,4.941642,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40672,0.693147,0.0,4.74,4.330733,1,1,1,1,0,0,5.361292,3.713572,31,5.420535,4.454347,0.000000,4.521789,0.000000,0
40673,0.000000,0.0,0.99,3.931826,0,1,0,0,0,0,5.267858,2.564949,20,5.220356,0.000000,0.000000,0.000000,0.000000,0
40674,4.382027,0.0,5.35,3.526361,0,1,1,1,0,0,8.087333,5.913503,8,8.421563,5.598422,0.000000,5.484797,0.000000,0
40675,0.000000,0.0,2.49,3.931826,0,1,1,1,0,0,5.342334,3.091042,4,5.379897,0.000000,0.000000,0.000000,0.000000,0


In [8]:
steam.shape

(40677, 19)

Splitting our train/test data + Normalising

In [9]:
# shuffling
steam = steam.sample(frac=1, random_state=42).reset_index(drop=True)

# splitting our data (80/20)
split = int(0.8 * len(steam))
train = steam.iloc[:split]
test = steam.iloc[split:]

In [10]:
# sep features and target
X_train = train.drop(columns=['Price']).values
y_train = train['Price'].values.reshape(-1, 1)

X_test = test.drop(columns=['Price']).values
y_test = test['Price'].values.reshape(-1, 1)

In [11]:
# min max scaling using our training data only
X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)

In [12]:
X_train_scaled = (X_train - X_min) / (X_max - X_min + 1e-8)
X_test_scaled = (X_test - X_min) / (X_max - X_min + 1e-8)

In [13]:
# adding bias column
X_train_scaled = np.hstack([X_train_scaled, np.ones((X_train_scaled.shape[0], 1))])
X_test_scaled = np.hstack([X_test_scaled, np.ones((X_test_scaled.shape[0], 1))])

print(X_train_scaled.shape)

(32541, 19)


MLP Implementation

In [14]:
## MLP setup

eta = 0.01
epochs = 500
p = X_train.shape[1]  # 19 (18 features + bias)
H = 4                  # hidden units

np.random.seed(1)
W1 = np.random.normal(0, 0.1, (p, H))
np.random.seed(1)
w2 = np.random.normal(0, 0.1, (H, 1))

In [15]:
# activation function
def relu(z):
    return np.maximum(0, z)


In [16]:
# gradient descent
errors = []

for epoch in range(epochs):
    grad_w2 = np.zeros_like(w2)
    grad_W1 = np.zeros_like(W1)
    loss = 0

    for i in range(X_train.shape[0]):
        x = X_train[i].reshape(-1, 1)
        yi = y_train[i]

        # forward pass
        # hidden layer
        h = relu(W1.T @ x)
        f = w2.T @ h

        # MSE loss
        loss += (f - yi) ** 2

        # gradients
        grad_w2 += (f - yi) * h
        indicator = (W1.T @ x > 0).astype(int)
        grad_W1 += (f - yi) * (x @ (w2 * indicator).T)

    # average gradients
    grad_w2 /= X_train.shape[0]
    grad_W1 /= X_train.shape[0]

    # update weights
    w2 -= eta * grad_w2
    W1 -= eta * grad_W1

    errors.append(loss / X_train.shape[0])

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, MSE Loss: {float(loss/X_train.shape[0]):.4f}")


/tmp/ipykernel_9662/194331899.py:37: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"Epoch {epoch}, MSE Loss: {float(loss/X_train.shape[0]):.4f}")


Epoch 0, MSE Loss: 39.9209
Epoch 50, MSE Loss: 11.7058
Epoch 100, MSE Loss: 11.3238
Epoch 150, MSE Loss: 11.0918
Epoch 200, MSE Loss: 10.9148
Epoch 250, MSE Loss: 17.2582
Epoch 300, MSE Loss: 13.4150
Epoch 350, MSE Loss: 12.4880
Epoch 400, MSE Loss: 12.0194
Epoch 450, MSE Loss: 11.6773


In [17]:
# test predicts
preds = []
for i in range(X_test.shape[0]):
    x = X_test[i].reshape(-1, 1)
    h = relu(W1.T @ x)
    f = w2.T @ h
    preds.append(float(f))

preds = np.array(preds)


/tmp/ipykernel_9662/185759057.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  preds.append(float(f))


In [18]:
# RMSE
rmse = np.sqrt(np.mean((preds - y_test) ** 2))
print(f"Test RMSE (log scale): {rmse:.4f}")

Test RMSE (log scale): 3.3899


Future steps:
- Sample-by-sample loop has been quite slow on 32k rows.
- For Phase II, we were thinking of vectorising it / looking for other ways to optimise run times and make model better because current errors are pretty high.

### Binary Purchase Classification (InGamePurchases)

This section adds a notebook-only pipeline for your player dataset (target = `InGamePurchases`).

Improvements focused on lowering probability RMSE:
- vectorized mini-batch MLP with Adam + L2 + dropout + early stopping
- Bayesian logistic regression baseline with posterior uncertainty
- train-only standardization, one-hot categorical encoding, stratified train/val/test split
- threshold tuning on validation set for stronger downstream filtering

In [19]:
import copy
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


@dataclass
class TrainingHistory:
    train_loss: List[float]
    val_loss: List[float]
    train_rmse: List[float]
    val_rmse: List[float]
    best_epoch: int


@dataclass
class StandardizationStats:
    mean: np.ndarray
    std: np.ndarray


def fit_standardizer(x_train: np.ndarray) -> StandardizationStats:
    mean = x_train.mean(axis=0, keepdims=True)
    std = x_train.std(axis=0, keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)
    return StandardizationStats(mean=mean, std=std)


def transform_standardizer(x: np.ndarray, stats: StandardizationStats) -> np.ndarray:
    return (x - stats.mean) / stats.std


def rmse_from_proba(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = y_true.reshape(-1, 1).astype(np.float64)
    y_prob = y_prob.reshape(-1, 1).astype(np.float64)
    return float(np.sqrt(np.mean((y_prob - y_true) ** 2)))


def binary_cross_entropy(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = y_true.reshape(-1, 1).astype(np.float64)
    y_prob = y_prob.reshape(-1, 1).astype(np.float64)
    eps = 1e-8
    y_prob = np.clip(y_prob, eps, 1.0 - eps)
    return float(-np.mean(y_true * np.log(y_prob) + (1.0 - y_true) * np.log(1.0 - y_prob)))


def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)

    acc = float((y_true == y_pred).mean())
    tp = float(((y_true == 1) & (y_pred == 1)).sum())
    fp = float(((y_true == 0) & (y_pred == 1)).sum())
    fn = float(((y_true == 1) & (y_pred == 0)).sum())

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2.0 * precision * recall / (precision + recall + 1e-8)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def evaluate_binary(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_prob = y_prob.reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)
    metrics = classification_metrics(y_true, y_pred)
    metrics["rmse"] = rmse_from_proba(y_true, y_prob)
    metrics["bce"] = binary_cross_entropy(y_true, y_prob)
    metrics["threshold"] = float(threshold)
    return metrics


def tune_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> Tuple[float, Dict[str, float]]:
    best_threshold = 0.5
    best_metrics = evaluate_binary(y_true, y_prob, threshold=0.5)
    best_f1 = best_metrics["f1"]

    for threshold in np.linspace(0.05, 0.95, 181):
        current = evaluate_binary(y_true, y_prob, threshold=float(threshold))
        if current["f1"] > best_f1:
            best_f1 = current["f1"]
            best_threshold = float(threshold)
            best_metrics = current

    return best_threshold, best_metrics


def stratified_train_val_test_split(
    x: np.ndarray,
    y: np.ndarray,
    train_frac: float = 0.7,
    val_frac: float = 0.15,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if not (0.0 < train_frac < 1.0 and 0.0 <= val_frac < 1.0 and train_frac + val_frac < 1.0):
        raise ValueError("Invalid split fractions.")

    rng = np.random.default_rng(seed)
    train_indices, val_indices, test_indices = [], [], []

    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)

        n_cls = len(cls_idx)
        n_train = int(np.floor(train_frac * n_cls))
        n_val = int(np.floor(val_frac * n_cls))

        if n_cls >= 3:
            n_train = max(1, n_train)
            n_val = max(1, n_val)
            if n_train + n_val >= n_cls:
                n_val = max(1, n_cls - n_train - 1)

        n_test = n_cls - n_train - n_val
        if n_test <= 0:
            n_test = 1
            if n_val > 1:
                n_val -= 1
            else:
                n_train = max(1, n_train - 1)

        train_indices.extend(cls_idx[:n_train].tolist())
        val_indices.extend(cls_idx[n_train:n_train + n_val].tolist())
        test_indices.extend(cls_idx[n_train + n_val:].tolist())

    train_idx = np.array(train_indices, dtype=int)
    val_idx = np.array(val_indices, dtype=int)
    test_idx = np.array(test_indices, dtype=int)

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return x[train_idx], y[train_idx], x[val_idx], y[val_idx], x[test_idx], y[test_idx]


def prepare_purchase_features(df: pd.DataFrame, target_col: str = "InGamePurchases"):
    if target_col not in df.columns:
        raise ValueError(f"Missing target column: {target_col}")

    work = df.copy()
    if "PlayerID" in work.columns:
        work = work.drop(columns=["PlayerID"])

    y = work[target_col].astype(int).to_numpy()
    x_df = work.drop(columns=[target_col])
    raw_feature_count = int(x_df.shape[1])

    cat_cols = [
        c for c in x_df.columns
        if pd.api.types.is_object_dtype(x_df[c]) or isinstance(x_df[c].dtype, pd.CategoricalDtype)
    ]

    x_df = pd.get_dummies(x_df, columns=cat_cols, drop_first=False, dtype=np.float64)
    x = x_df.to_numpy(dtype=np.float64)

    return x, y, x_df.columns.tolist(), raw_feature_count

In [20]:
class NumpyMLPClassifier:
    """Two-hidden-layer MLP for binary classification implemented from scratch."""

    def __init__(
        self,
        input_dim: int,
        hidden1: int = 128,
        hidden2: int = 64,
        lr: float = 1e-3,
        l2: float = 1e-4,
        dropout: float = 0.10,
        beta1: float = 0.9,
        beta2: float = 0.999,
        adam_eps: float = 1e-8,
        seed: int = 42,
    ):
        self.rng = np.random.default_rng(seed)
        self.lr = lr
        self.l2 = l2
        self.dropout = float(np.clip(dropout, 0.0, 0.7))
        self.beta1 = beta1
        self.beta2 = beta2
        self.adam_eps = adam_eps
        self.step_count = 0

        self.params = {
            "W1": self._he_init(input_dim, hidden1),
            "b1": np.zeros((1, hidden1), dtype=np.float64),
            "W2": self._he_init(hidden1, hidden2),
            "b2": np.zeros((1, hidden2), dtype=np.float64),
            "W3": self._he_init(hidden2, 1),
            "b3": np.zeros((1, 1), dtype=np.float64),
        }

        self.opt_state = {}
        for k, v in self.params.items():
            self.opt_state[f"m_{k}"] = np.zeros_like(v)
            self.opt_state[f"v_{k}"] = np.zeros_like(v)

    def _he_init(self, fan_in: int, fan_out: int) -> np.ndarray:
        std = np.sqrt(2.0 / max(1, fan_in))
        return self.rng.normal(0.0, std, size=(fan_in, fan_out)).astype(np.float64)

    @staticmethod
    def _relu(x: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, x)

    @staticmethod
    def _relu_grad(x: np.ndarray) -> np.ndarray:
        return (x > 0.0).astype(np.float64)

    @staticmethod
    def _sigmoid(x: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30.0, 30.0)))

    def _apply_dropout(self, a: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if self.dropout <= 0.0:
            return a, np.ones_like(a)
        keep_prob = 1.0 - self.dropout
        mask = (self.rng.random(a.shape) < keep_prob).astype(np.float64) / keep_prob
        return a * mask, mask

    def _forward(self, x: np.ndarray, train: bool) -> Dict[str, np.ndarray]:
        z1 = x @ self.params["W1"] + self.params["b1"]
        a1 = self._relu(z1)

        if train:
            a1, mask1 = self._apply_dropout(a1)
        else:
            mask1 = np.ones_like(a1)

        z2 = a1 @ self.params["W2"] + self.params["b2"]
        a2 = self._relu(z2)

        if train:
            a2, mask2 = self._apply_dropout(a2)
        else:
            mask2 = np.ones_like(a2)

        z3 = a2 @ self.params["W3"] + self.params["b3"]
        y_prob = self._sigmoid(z3)

        return {
            "x": x,
            "z1": z1,
            "a1": a1,
            "mask1": mask1,
            "z2": z2,
            "a2": a2,
            "mask2": mask2,
            "y_prob": y_prob,
        }

    def _backward(self, cache: Dict[str, np.ndarray], y_true: np.ndarray) -> Dict[str, np.ndarray]:
        y_true = y_true.reshape(-1, 1).astype(np.float64)
        m = y_true.shape[0]

        dz3 = cache["y_prob"] - y_true
        dW3 = (cache["a2"].T @ dz3) / m + self.l2 * self.params["W3"]
        db3 = np.sum(dz3, axis=0, keepdims=True) / m

        da2 = dz3 @ self.params["W3"].T
        da2 *= cache["mask2"]
        dz2 = da2 * self._relu_grad(cache["z2"])
        dW2 = (cache["a1"].T @ dz2) / m + self.l2 * self.params["W2"]
        db2 = np.sum(dz2, axis=0, keepdims=True) / m

        da1 = dz2 @ self.params["W2"].T
        da1 *= cache["mask1"]
        dz1 = da1 * self._relu_grad(cache["z1"])
        dW1 = (cache["x"].T @ dz1) / m + self.l2 * self.params["W1"]
        db1 = np.sum(dz1, axis=0, keepdims=True) / m

        return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "W3": dW3, "b3": db3}

    def _step(self, grads: Dict[str, np.ndarray]) -> None:
        self.step_count += 1

        for k in self.params:
            m = self.opt_state[f"m_{k}"]
            v = self.opt_state[f"v_{k}"]

            m = self.beta1 * m + (1.0 - self.beta1) * grads[k]
            v = self.beta2 * v + (1.0 - self.beta2) * (grads[k] ** 2)

            m_hat = m / (1.0 - self.beta1 ** self.step_count)
            v_hat = v / (1.0 - self.beta2 ** self.step_count)

            self.params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.adam_eps)

            self.opt_state[f"m_{k}"] = m
            self.opt_state[f"v_{k}"] = v

    def predict_proba(self, x: np.ndarray) -> np.ndarray:
        return self._forward(x, train=False)["y_prob"]

    def predict(self, x: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(x).reshape(-1) >= threshold).astype(int)

    def fit(
        self,
        x_train: np.ndarray,
        y_train: np.ndarray,
        x_val: np.ndarray,
        y_val: np.ndarray,
        epochs: int = 250,
        batch_size: int = 256,
        patience: int = 20,
        min_delta: float = 1e-4,
        verbose: bool = True,
    ) -> TrainingHistory:
        history = TrainingHistory(train_loss=[], val_loss=[], train_rmse=[], val_rmse=[], best_epoch=0)
        n = x_train.shape[0]

        best_val_loss = np.inf
        best_epoch = 0
        stale_epochs = 0
        best_params = copy.deepcopy(self.params)
        best_opt_state = copy.deepcopy(self.opt_state)

        for epoch in range(1, epochs + 1):
            idx = self.rng.permutation(n)

            for start in range(0, n, batch_size):
                batch = idx[start:start + batch_size]
                cache = self._forward(x_train[batch], train=True)
                grads = self._backward(cache, y_train[batch])
                self._step(grads)

            train_prob = self.predict_proba(x_train)
            val_prob = self.predict_proba(x_val)

            train_loss = binary_cross_entropy(y_train, train_prob)
            val_loss = binary_cross_entropy(y_val, val_prob)
            train_rmse = rmse_from_proba(y_train, train_prob)
            val_rmse = rmse_from_proba(y_val, val_prob)

            history.train_loss.append(train_loss)
            history.val_loss.append(val_loss)
            history.train_rmse.append(train_rmse)
            history.val_rmse.append(val_rmse)

            if val_loss < (best_val_loss - min_delta):
                best_val_loss = val_loss
                best_epoch = epoch
                stale_epochs = 0
                best_params = copy.deepcopy(self.params)
                best_opt_state = copy.deepcopy(self.opt_state)
            else:
                stale_epochs += 1

            if verbose and (epoch == 1 or epoch % 10 == 0):
                print(
                    f"Epoch {epoch:03d} | "
                    f"train_bce={train_loss:.4f} val_bce={val_loss:.4f} "
                    f"train_rmse={train_rmse:.4f} val_rmse={val_rmse:.4f}"
                )

            if stale_epochs >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}.")
                break

        self.params = best_params
        self.opt_state = best_opt_state
        history.best_epoch = best_epoch
        return history


class BayesianLogisticRegression:
    """Bayesian logistic regression with Gaussian prior + Laplace approximation."""

    def __init__(self, prior_precision: float = 1.0, max_iter: int = 80, tol: float = 1e-6, fit_intercept: bool = True):
        self.prior_precision = prior_precision
        self.max_iter = max_iter
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.w_map = None
        self.posterior_cov = None

    @staticmethod
    def _sigmoid(x: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30.0, 30.0)))

    def _with_bias(self, x: np.ndarray) -> np.ndarray:
        if not self.fit_intercept:
            return x
        return np.hstack([x, np.ones((x.shape[0], 1), dtype=np.float64)])

    def fit(self, x: np.ndarray, y: np.ndarray) -> None:
        x = self._with_bias(x.astype(np.float64))
        y = y.reshape(-1).astype(np.float64)

        _, d = x.shape
        w = np.zeros(d, dtype=np.float64)
        prior = self.prior_precision * np.eye(d, dtype=np.float64)

        for _ in range(self.max_iter):
            p = self._sigmoid(x @ w)
            r = p * (1.0 - p)
            hessian = x.T @ (x * r[:, None]) + prior
            grad = x.T @ (p - y) + self.prior_precision * w

            try:
                step = np.linalg.solve(hessian, grad)
            except np.linalg.LinAlgError:
                step = np.linalg.lstsq(hessian, grad, rcond=None)[0]

            w_new = w - step
            if np.linalg.norm(w_new - w) < self.tol:
                w = w_new
                break
            w = w_new

        p = self._sigmoid(x @ w)
        r = p * (1.0 - p)
        hessian = x.T @ (x * r[:, None]) + prior

        self.w_map = w
        try:
            self.posterior_cov = np.linalg.inv(hessian)
        except np.linalg.LinAlgError:
            self.posterior_cov = np.linalg.pinv(hessian)

    def predict_proba(self, x: np.ndarray, n_samples: int = 300, seed: int = 42):
        if self.w_map is None or self.posterior_cov is None:
            raise ValueError("Model must be fitted before prediction.")

        x = self._with_bias(x.astype(np.float64))
        rng = np.random.default_rng(seed)

        samples = rng.multivariate_normal(self.w_map, self.posterior_cov, size=n_samples)
        probs = self._sigmoid(x @ samples.T)

        mean_prob = probs.mean(axis=1)
        lower = np.quantile(probs, 0.025, axis=1)
        upper = np.quantile(probs, 0.975, axis=1)
        return mean_prob, lower, upper

In [21]:
def run_purchase_experiment(
    df: pd.DataFrame,
    target_col: str = "InGamePurchases",
    seed: int = 42,
    epochs: int = 250,
    batch_size: int = 256,
    verbose: bool = True,
):
    x, y, feature_names, raw_feature_count = prepare_purchase_features(df, target_col=target_col)

    x_train, y_train, x_val, y_val, x_test, y_test = stratified_train_val_test_split(
        x, y, train_frac=0.7, val_frac=0.15, seed=seed
    )

    stats = fit_standardizer(x_train)
    x_train_s = transform_standardizer(x_train, stats)
    x_val_s = transform_standardizer(x_val, stats)
    x_test_s = transform_standardizer(x_test, stats)

    mlp = NumpyMLPClassifier(
        input_dim=x_train_s.shape[1],
        hidden1=128,
        hidden2=64,
        lr=1e-3,
        l2=1e-4,
        dropout=0.10,
        seed=seed,
    )

    history = mlp.fit(
        x_train=x_train_s,
        y_train=y_train,
        x_val=x_val_s,
        y_val=y_val,
        epochs=epochs,
        batch_size=batch_size,
        patience=20,
        min_delta=1e-4,
        verbose=verbose,
    )

    mlp_val_prob = mlp.predict_proba(x_val_s).reshape(-1)
    mlp_threshold, _ = tune_threshold(y_val, mlp_val_prob)
    mlp_test_prob = mlp.predict_proba(x_test_s).reshape(-1)
    mlp_metrics = evaluate_binary(y_test, mlp_test_prob, threshold=mlp_threshold)

    blr = BayesianLogisticRegression(prior_precision=1.0, max_iter=80, tol=1e-6, fit_intercept=True)
    blr.fit(x_train_s, y_train)

    blr_val_prob, _, _ = blr.predict_proba(x_val_s, n_samples=300, seed=seed)
    blr_threshold, _ = tune_threshold(y_val, blr_val_prob)
    blr_test_prob, blr_lower, blr_upper = blr.predict_proba(x_test_s, n_samples=300, seed=seed + 1)
    blr_metrics = evaluate_binary(y_test, blr_test_prob, threshold=blr_threshold)

    results = {
        "dataset": {
            "n_rows": int(x.shape[0]),
            "n_raw_features": raw_feature_count,
            "n_features_after_encoding": int(x.shape[1]),
            "positive_rate": float(y.mean()),
            "n_encoded_feature_names": len(feature_names),
        },
        "training": {
            "best_epoch": int(history.best_epoch),
            "best_val_bce": float(np.min(history.val_loss)),
            "best_val_rmse": float(np.min(history.val_rmse)),
        },
        "mlp": mlp_metrics,
        "bayesian_logistic": {
            **blr_metrics,
            "mean_posterior_interval_width": float(np.mean(blr_upper - blr_lower)),
        },
    }

    return results


def format_results(results):
    lines = []
    ds = results["dataset"]
    lines.append("Dataset")
    lines.append(
        f"  rows={ds['n_rows']}, raw_features={ds['n_raw_features']}, encoded_features={ds['n_features_after_encoding']}, positive_rate={ds['positive_rate']:.4f}"
    )

    tr = results["training"]
    lines.append("Training")
    lines.append(
        f"  best_epoch={tr['best_epoch']}, best_val_bce={tr['best_val_bce']:.4f}, best_val_rmse={tr['best_val_rmse']:.4f}"
    )

    lines.append("MLP Test Metrics")
    for k in ["accuracy", "precision", "recall", "f1", "rmse", "bce", "threshold"]:
        lines.append(f"  {k}={results['mlp'][k]:.4f}")

    lines.append("Bayesian Logistic Test Metrics")
    for k in ["accuracy", "precision", "recall", "f1", "rmse", "bce", "threshold", "mean_posterior_interval_width"]:
        lines.append(f"  {k}={results['bayesian_logistic'][k]:.4f}")

    return "\n".join(lines)

In [22]:
# Example usage with your player dataset:
# player_df = pd.read_csv("your_player_data.csv")
# results = run_purchase_experiment(
#     player_df,
#     target_col="InGamePurchases",
#     seed=42,
#     epochs=250,
#     batch_size=256,
#     verbose=True,
# )
# print(format_results(results))

# Optional quick check if current in-memory dataframe already has this target:
if 'steam' in globals() and isinstance(steam, pd.DataFrame) and 'InGamePurchases' in steam.columns:
    results = run_purchase_experiment(steam, target_col='InGamePurchases', epochs=120, batch_size=256, verbose=False)
    print(format_results(results))
else:
    print("Load your player dataset into `player_df` and run the example code above.")

Load your player dataset into `player_df` and run the example code above.


### RMSE Rerun (Price Regression, steam_clean.csv)

This project's target is `Price`, so this is **regression** (not 0/1 buy classification).

This block compares on the same 80/20 split (`random_state=42`):
- Baseline notebook MLP update rule (vectorized-equivalent)
- Linear/Ridge regression
- MLP regressor
- HistGradientBoostingRegressor

In [27]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

csv_path = "/content/steam_clean.csv"

df_cmp = pd.read_csv(csv_path)
df_cmp = df_cmp.sample(frac=1, random_state=42).reset_index(drop=True)

split = int(0.8 * len(df_cmp))
train_cmp = df_cmp.iloc[:split]
test_cmp = df_cmp.iloc[split:]

X_train_cmp = train_cmp.drop(columns=["Price"]).to_numpy(dtype=np.float64)
y_train_cmp = train_cmp["Price"].to_numpy(dtype=np.float64)
X_test_cmp = test_cmp.drop(columns=["Price"]).to_numpy(dtype=np.float64)
y_test_cmp = test_cmp["Price"].to_numpy(dtype=np.float64)


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


# 1) Baseline notebook-style MLP (vectorized equivalent for speed).
eta = 0.01
epochs = 500
p_dim = X_train_cmp.shape[1]
H = 4

np.random.seed(1)
W1 = np.random.normal(0, 0.1, (p_dim, H))
np.random.seed(1)
w2 = np.random.normal(0, 0.1, (H, 1))
y_train_col = y_train_cmp.reshape(-1, 1)

for _ in range(epochs):
    z1 = X_train_cmp @ W1
    h = np.maximum(0.0, z1)
    f = h @ w2
    err = f - y_train_col

    grad_w2 = (h.T @ err) / X_train_cmp.shape[0]
    indicator = (z1 > 0).astype(np.float64)
    grad_W1 = (X_train_cmp.T @ (err * indicator * w2.T)) / X_train_cmp.shape[0]

    w2 -= eta * grad_w2
    W1 -= eta * grad_W1

baseline_pred = (np.maximum(0.0, X_test_cmp @ W1) @ w2).reshape(-1)
baseline_rmse = rmse(y_test_cmp, baseline_pred)


# 2) Stronger regression models.
models = {
    "linear": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0, random_state=42)),
    ]),
    "mlp_regressor": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            hidden_layer_sizes=(128, 64),
            random_state=42,
            max_iter=600,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
        )),
    ]),
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        random_state=42,
        learning_rate=0.05,
        max_depth=8,
        max_iter=400,
        l2_regularization=0.01,
    ),
}

results = {"baseline_notebook_mlp": baseline_rmse}
for name, model in models.items():
    model.fit(X_train_cmp, y_train_cmp)
    pred = model.predict(X_test_cmp)
    results[name] = rmse(y_test_cmp, pred)

for k, v in results.items():
    print(f"{k}: {v:.6f}")

best_model = min(results, key=results.get)
best_rmse = results[best_model]
improvement = baseline_rmse - best_rmse
improvement_pct = (improvement / baseline_rmse) * 100.0

print()
print("Best model:", best_model)
print(f"Best RMSE: {best_rmse:.6f}")
print(f"Improvement vs baseline: {improvement:.6f} ({improvement_pct:.2f}%)")

baseline_notebook_mlp: 3.388250
linear: 0.405974
ridge: 0.405974
mlp_regressor: 0.374315
hist_gradient_boosting: 0.341865

Best model: hist_gradient_boosting
Best RMSE: 0.341865
Improvement vs baseline: 3.046385 (89.91%)


### Why This Got Lower

`Price` behaves like a continuous target on a transformed scale, so regression models that capture nonlinearity generally outperform the tiny baseline network.

Why RMSE dropped below `0.405974`:
1. The baseline notebook MLP is very small (`H=4`) and underfits.
2. The improved models use better optimization and regularization (standardization + early stopping/L2).
3. HistGradientBoosting captures nonlinear interactions among engagement/review/content features.
4. All models are evaluated on the same split for a fair comparison.

Interpretation note:
- If your target is `Price`, your metric should be regression RMSE/MAE.
- `0/1` buy prediction would be a separate classification task with target like `InGamePurchases`.